In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch
import pandas as pd
import numpy as np
import shutil
import os

!pip install transformers torch pandas numpy
# import colab stuff, skip these if work on local venv

In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
    folder_path = "/content/drive/MyDrive/my_data_folder/"
except ImportError:
    folder_path = "./processed_data"

In [ ]:
X_train_path = os.path.join(folder_path, "X_train.csv")
X_test_path = os.path.join(folder_path, "X_test.csv")
X_val_path = os.path.join(folder_path, "X_val.csv")

print("Data paths set. Checking if files exist...")
for f in [X_train_path, X_test_path, X_val_path]:
    print(f"{f}: {'Found' if os.path.exists(f) else 'Not Found'}")

Data paths set. Checking if files exist...
./processed_data\X_train.csv: Found
./processed_data\X_test.csv: Found
./processed_data\X_val.csv: Found


In [ ]:
# Load the datasets using the Drive paths
X_train = pd.read_csv(X_train_path)
X_test = pd.read_csv(X_test_path)
X_val = pd.read_csv(X_val_path)

X_train.head()

,value,"Temperature, K","Pressure, kPa",Component 1,Component 2,Smiles 1,Smiles 2,mol1,mol2,"('BalabanJ', <class 'numpy.float64'>)",...,"property_Molar heat capacity at constant volume, J/K/mol","property_Molar volume, m3/mol",property_Refractive index (Na D-line),"property_Specific volume, m3/kg","property_Speed of sound, m/s","property_Thermal conductivity, W/m/K","property_Tracer diffusion coefficient, m2/s","property_Viscosity, Pa*s",phase_Gas,phase_Liquid
0,695.300000,362.710,3000.0,decane,ethanol,CCCCCCCCCC,CCO,<rdkit.Chem.rdchem.Mol object at 0x0000026C68E...,<rdkit.Chem.rdchem.Mol object at 0x0000026CCF5...,0.948491,...,False,False,False,False,False,False,False,False,False,True
1,855.400000,298.150,35000.0,"3,7-dimethyl-1,6-octadien-3-ol",propan-1-ol,CC(=CCCC(C)(C=C)O)C,CCCO,<rdkit.Chem.rdchem.Mol object at 0x0000026C68D...,<rdkit.Chem.rdchem.Mol object at 0x0000026CABA...,0.992385,...,False,False,False,False,False,False,False,False,False,True
2,0.000791,278.150,45000.0,hexan-1-ol,hexane,CCCCCCO,CCCCCC,<rdkit.Chem.rdchem.Mol object at 0x0000026C68A...,<rdkit.Chem.rdchem.Mol object at 0x0000026C678...,0.918955,...,False,False,False,False,False,False,False,False,True,False
3,0.000018,323.216,398.6,carbon dioxide,nitrogen,C(=O)=O,N#N,<rdkit.Chem.rdchem.Mol object at 0x0000026C68F...,<rdkit.Chem.rdchem.Mol object at 0x0000026CD36...,0.984555,...,False,False,False,False,False,False,False,True,True,False
4,651.800000,353.150,10000.0,ethanol,heptane,CCO,CCCCCCC,<rdkit.Chem.rdchem.Mol object at 0x0000026C68F...,<rdkit.Chem.rdchem.Mol object at 0x0000026CD15...,0.404909,...,False,False,False,False,False,False,False,False,False,True


In [6]:
# X_train = pd.read_csv("X_train.csv")
# X_test = pd.read_csv("X_test.csv")
# X_val = pd.read_csv("X_val.csv")

In [8]:
smiles1_train = X_train["Smiles 1"].tolist()
smiles2_train = X_train["Smiles 2"].tolist()

smiles1_val = X_val["Smiles 1"].tolist()
smiles2_val = X_val["Smiles 2"].tolist()

smiles1_test = X_test["Smiles 1"].tolist()
smiles2_test = X_test["Smiles 2"].tolist()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("UdS-LSV/smole-bert")
model = AutoModel.from_pretrained("UdS-LSV/smole-bert")
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


def get_embeddings(smiles_list, batch_size=1024):
    all_embeddings = []
    n_batches = len(smiles_list) // batch_size + 1
    for i in range(0, len(smiles_list), batch_size):
        batch = smiles_list[i: i + batch_size]
        encoded = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128,
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            output = model(**encoded)
        # CLS token embedding as molecular representation
        embeddings = output.last_hidden_state[:, 0, :]
        all_embeddings.append(embeddings.cpu().numpy())
        if (i // batch_size + 1) % 10 == 0:
            print(f"Batch {i // batch_size + 1}/{n_batches}")
    return np.concatenate(all_embeddings, axis=0)


# Compute and save
E1_train = get_embeddings(smiles1_train)
E2_train = get_embeddings(smiles2_train)

E1_val = get_embeddings(smiles1_val)
E2_val = get_embeddings(smiles2_val)

E1_test = get_embeddings(smiles1_test)
E2_test = get_embeddings(smiles2_test)

np.save("E1_train.npy", E1_train)
np.save("E2_train.npy", E2_train)

np.save("E1_val.npy", E1_val)
np.save("E2_val.npy", E2_val)

np.save("E1_test.npy", E1_test)
np.save("E2_test.npy", E2_test)

Loading weights: 100%|██████████| 101/101 [00:00<00:00, 5796.33it/s]
BertModel LOAD REPORT from: UdS-LSV/smole-bert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Batch 10/84
Batch 20/84
Batch 30/84
Batch 40/84
Batch 50/84
Batch 60/84
Batch 70/84
Batch 80/84
Batch 10/84
Batch 20/84
Batch 30/84
Batch 40/84
Batch 50/84
Batch 60/84
Batch 70/84
Batch 80/84
Batch 10/11
Batch 10/11
Batch 10/11
Batch 10/11


In [11]:
# List of files to save
files_to_save = [
    "E1_train.npy",
    "E2_train.npy",
    "E1_val.npy",
    "E2_val.npy",
    "E1_test.npy",
    "E2_test.npy",
]

# Copy each file to your specified Google Drive folder
for f in files_to_save:
    if os.path.exists(f):
        shutil.copy(f, os.path.join(folder_path, f))
        print(f"Saved {f} to {folder_path}")
    else:
        print(f"File {f} not found in local storage.")

Saved E1_train.npy to ./processed_data
Saved E2_train.npy to ./processed_data
Saved E1_val.npy to ./processed_data
Saved E2_val.npy to ./processed_data
Saved E1_test.npy to ./processed_data
Saved E2_test.npy to ./processed_data
